# 01 · Multi-Agent Architectures — 课程索引



# 01 · Multi-Agent Architectures — 动手 Demo


In [1]:
import nest_asyncio; nest_asyncio.apply()
import queue, threading, asyncio, time, uuid, random, copy
from collections import defaultdict
from dataclasses import dataclass, field
from typing import Any

# ── 模拟 LLM / 向量检索 ─────────────────────────────────────────────────────
CORPUS = [
    "RAG 通过检索外部文档来扩充 LLM 的上下文。",
    "向量数据库支持语义相似度搜索。",
    "Reranker 用交叉编码器对候选文档重新打分。",
    "幂等设计让系统在重试时不产生副作用。",
    "Backpressure 防止下游服务因流量过大而崩溃。",
    "可观测性包括 Metrics、Traces 和 Logs 三大支柱。",
    "Graceful degradation 让系统在部分故障时仍能返回有用结果。",
    "Hub-and-Spoke 模式由 Orchestrator 统一分派任务。",
    "Blackboard 模式通过共享工作区实现多 Agent 协作。",
    "消息队列解耦生产者和消费者，支持异步处理。",
]

def fake_retrieve(query: str, top_k: int = 5, latency: float = 0.3) -> list[dict]:
    """模拟向量检索，返回 top_k 相关文档"""
    time.sleep(latency)
    scored = [(doc, random.uniform(0.5, 1.0)) for doc in CORPUS]
    scored.sort(key=lambda x: -x[1])
    return [{"text": doc, "score": round(s, 3)} for doc, s in scored[:top_k]]

def fake_rerank(query: str, hits: list[dict], top_k: int = 3, latency: float = 0.2) -> list[dict]:
    """模拟 cross-encoder reranker"""
    time.sleep(latency)
    reranked = copy.deepcopy(hits)
    for h in reranked:
        h["score"] = round(h["score"] * random.uniform(0.9, 1.1), 3)
    reranked.sort(key=lambda x: -x["score"])
    return reranked[:top_k]

def fake_generate(question: str, context: list[dict], latency: float = 0.5) -> str:
    """模拟 LLM 生成"""
    time.sleep(latency)
    snippets = " ".join(h["text"][:30] for h in context[:2])
    return f"[答案] 关于「{question[:20]}」：{snippets}..."

def fake_verify(answer: str, context: list[dict], latency: float = 0.15) -> dict:
    """模拟答案校验"""
    time.sleep(latency)
    return {"grounded": random.random() > 0.2, "confidence": round(random.uniform(0.7, 1.0), 2)}

print("✅ 模拟工具加载完毕")

✅ 模拟工具加载完毕


---
## Part 1 · 单体 Agent 的局限

所有逻辑塞在一个函数里：检索 → 重排 → 生成 → 校验，**串行执行**。

```
用户 ──▶ [ monolithic_agent() ] ──▶ 回答
            ├─ retrieve  (0.3s)
            ├─ rerank    (0.2s)
            ├─ generate  (0.5s)
            └─ verify    (0.15s)
               ──────────────────
               总计 ≈ 1.15s（串行叠加）
```

In [2]:
def monolithic_agent(query: str) -> dict:
    """单体 Agent：所有步骤串行执行"""
    t0 = time.perf_counter()

    hits    = fake_retrieve(query, top_k=5)
    topk    = fake_rerank(query, hits, top_k=3)
    answer  = fake_generate(query, topk)
    verdict = fake_verify(answer, topk)

    elapsed = round(time.perf_counter() - t0, 3)
    return {"answer": answer, "verdict": verdict, "latency_s": elapsed}

# ── 运行 ─────────────────────────────────────────────────────────────────────
result = monolithic_agent("什么是 RAG？")
print(f"延迟：{result['latency_s']}s")
print(f"校验：{result['verdict']}")
print(f"答案：{result['answer'][:80]}...")

延迟：1.151s
校验：{'grounded': True, 'confidence': 0.71}
答案：[答案] 关于「什么是 RAG？」：Reranker 用交叉编码器对候选文档重新打分。 Blackboard 模式通过共享工作区实现多 Agent ......


> **观察**：延迟 ≈ 各步骤之和。任何一步慢，整体就慢。
> 接下来我们拆分为多 Agent，观察延迟和可维护性的变化。

## 📖 讲义

# Part 1
## 引言：为什么需要多 Agent？

---

# 单体 Agent 的局限

```
用户 ──▶ [ 一个 LLM + 若干工具 ] ──▶ 回答
```

**在以下场景会遇到瓶颈：**

- **规模瓶颈**：检索 × 生成 × 验证串行，延迟叠加
- **职责模糊**：一个 Agent 承担过多角色，难以优化
- **无法独立扩容**：检索并发高但生成成本高，无法分别调配
- **故障扩散**：任一环节失败导致整体不可用
- **迭代困难**：换一个 Reranker 需要改动整个 Agent

---

# 多 Agent 带来什么？

```
用户
 │
 ▼
Orchestrator
 ├──▶ Retriever  (高并发，独立扩容)
 ├──▶ Reranker   (可替换实现)
 ├──▶ Generator  (成本精细控制)
 └──▶ Verifier   (独立故障隔离)
```

- **职责分离** — 每个 Agent 专注一件事，易测试、易替换
- **独立扩容** — 各服务按需弹性，成本可控
- **故障隔离** — Reranker 挂了可降级，不影响其他路径
- **并行执行** — 多个 Agent 同时工作，总延迟更短

---

---
## Part 2 · Pipeline 模式（队列 + Worker 线程）

用 Python 内置 `queue.Queue` 模拟 Redis 列表，每个 Worker 是一个线程。

```
Orchestrator
     │ put(job)
     ▼
q_retrieve ──▶ RetrieverWorker
                    │ put(job+hits)
                    ▼
              q_rerank ──▶ RerankerWorker
                                │ put(job+topk)
                                ▼
                          q_generate ──▶ GeneratorWorker
                                              │ put(result)
                                              ▼
                                         results[job_id]
```

In [3]:
# ── 队列定义 ─────────────────────────────────────────────────────────────────
q_retrieve = queue.Queue()
q_rerank   = queue.Queue()
q_generate = queue.Queue()
results    = {}          # job_id → result（生产中用 Redis / DB）
_stop      = threading.Event()

# ── Worker 定义 ───────────────────────────────────────────────────────────────
def retriever_worker():
    while not _stop.is_set():
        try:
            job = q_retrieve.get(timeout=0.2)
        except queue.Empty:
            continue
        print(f"  [Retriever] processing {job['request_id'][:8]}...")
        job = {**job, "hits": fake_retrieve(job["query"], latency=0.3)}
        q_rerank.put(job)
        q_retrieve.task_done()

def reranker_worker():
    while not _stop.is_set():
        try:
            job = q_rerank.get(timeout=0.2)
        except queue.Empty:
            continue
        print(f"  [Reranker]  processing {job['request_id'][:8]}...")
        job = {**job, "topk": fake_rerank(job["query"], job["hits"], latency=0.2)}
        q_generate.put(job)
        q_rerank.task_done()

def generator_worker():
    while not _stop.is_set():
        try:
            job = q_generate.get(timeout=0.2)
        except queue.Empty:
            continue
        print(f"  [Generator] processing {job['request_id'][:8]}...")
        answer  = fake_generate(job["query"], job["topk"], latency=0.5)
        verdict = fake_verify(answer, job["topk"], latency=0.15)
        results[job["request_id"]] = {
            "answer": answer,
            "verdict": verdict,
            "finished_at": time.perf_counter(),
        }
        q_generate.task_done()

# ── 启动 Workers ──────────────────────────────────────────────────────────────
_stop.clear()
workers = [
    threading.Thread(target=retriever_worker, daemon=True),
    threading.Thread(target=reranker_worker,  daemon=True),
    threading.Thread(target=generator_worker, daemon=True),
]
for w in workers:
    w.start()

print("✅ 3 个 Worker 已启动（Retriever / Reranker / Generator）")

✅ 3 个 Worker 已启动（Retriever / Reranker / Generator）


  [Retriever] processing a6759e58...
  [Retriever] processing 75ac7b2a...
  [Reranker]  processing a6759e58...
  [Generator] processing a6759e58...
  [Retriever] processing 70476a88...  [Reranker]  processing 75ac7b2a...

  [Reranker]  processing 70476a88...
  [Generator] processing 75ac7b2a...
  [Generator] processing 70476a88...


In [4]:
# ── 提交多个查询，观察并发流水线 ────────────────────────────────────────────
queries = ["什么是 RAG？", "幂等设计是什么？", "如何实现可观测性？"]
job_ids = []
t_start = time.perf_counter()

for q in queries:
    job_id = str(uuid.uuid4())
    job_ids.append((job_id, q, time.perf_counter()))
    q_retrieve.put({"request_id": job_id, "query": q})
    print(f"➡ 提交：{q[:20]}  [{job_id[:8]}]")

# 等待所有结果
print("\n⏳ 等待结果...")
for job_id, query, t_submit in job_ids:
    while job_id not in results:
        time.sleep(0.05)
    r = results[job_id]
    latency = round(r["finished_at"] - t_submit, 3)
    print(f"  ✅ [{job_id[:8]}] {query[:20]} → {latency}s | grounded={r['verdict']['grounded']}")

total = round(time.perf_counter() - t_start, 3)
print(f"\n总耗时：{total}s（3 个查询并行流水线）")

# 停止 Workers
_stop.set()

➡ 提交：什么是 RAG？  [a6759e58]
➡ 提交：幂等设计是什么？  [75ac7b2a]
➡ 提交：如何实现可观测性？  [70476a88]

⏳ 等待结果...
  ✅ [a6759e58] 什么是 RAG？ → 1.155s | grounded=False
  ✅ [75ac7b2a] 幂等设计是什么？ → 1.807s | grounded=True
  ✅ [70476a88] 如何实现可观测性？ → 2.458s | grounded=True

总耗时：2.492s（3 个查询并行流水线）


> **关键观察**：
> - 三个查询**同时**在流水线中流动，总时间 ≈ 单次时间（而非 3×）
> - 每个 Worker 职责单一，替换 Reranker 不影响其他两个
> - 队列天然起到**背压缓冲**作用

## 📖 讲义

# Part 2
## 核心概念：设计原则与架构模式

---

# 多 Agent 的四大设计原则

<div class="columns">
<div>

### 1. 职责分离 SRP
每个 Agent 只做一件明确的事
- Retriever → 只负责检索
- Reranker  → 只负责重排
- Generator → 只负责生成
- Verifier  → 只负责校验

</div>
<div>

### 2. 松耦合通信
- 通过消息/队列/API 交换数据
- 不直接调用对方内部函数
- 接口契约（schema）稳定

### 3. 可独立扩展
- 每个 Agent 独立部署
- 按负载单独扩缩容

### 4. 可替换性
- 换掉一个 Agent 不影响其他

</div>
</div>

---

# 架构模式 1：Pipeline（流水线）

```
请求
 │
 ▼
[Retriever] ──▶ [Reranker] ──▶ [Generator] ──▶ [Verifier] ──▶ 响应
```

**特点**
- 线性、顺序处理，易于理解与调试
- 每个阶段输出即下一阶段输入
- 适合：文档 QA、内容审核、数据清洗流水线

**实现方式**
- 消息队列（Redis / Kafka）：每个 Worker 消费上游队列，写入下游队列
- 同步链式调用：Orchestrator 依次调用各服务

---

# 架构模式 2：Hub-and-Spoke（协调器）

```
                ┌─── Retriever ──┐
                │                │
用户 ──▶ Orchestrator ─── Reranker ───▶ 合成回复
                │                │
                └─── Generator ──┘
```

**特点**
- Orchestrator 负责任务分派与结果合成
- 支持并行调用多个 Agent
- 适合：复杂问答、多步推理、工作流编排

**注意**
- Orchestrator 成为单点，需做好容错与限流
- 建议用异步 + 回调，避免阻塞

---

# 架构模式 3：Blackboard（共享工作区）

```
          写入              读取
Agent A ──────▶ Blackboard ◀────── Agent C
Agent B ──────▶     │     ◀────── Agent D
                    │
              变更通知（事件）
```

**特点**
- 多个 Agent 共享同一"黑板"（数据库/缓存/对象存储）
- 通过事件/轮询感知变化，异步推进
- 适合：迭代协作、多模型协商、长任务

**挑战**
- 需要设计好写入合约与冲突处理
- 数据清理策略（TTL / GC）

---

# 三种模式对比

| 维度 | Pipeline | Hub-and-Spoke | Blackboard |
|------|----------|---------------|------------|
| 复杂度 | 低 | 中 | 高 |
| 调试难度 | 易 | 中 | 难 |
| 并行能力 | 低 | 高 | 高 |
| 适合任务 | 线性 | 多角色协作 | 迭代协商 |
| 典型实现 | Redis Queue | FastAPI + gRPC | DB + Pub-Sub |

> **建议**：从 Pipeline 起步，按需演进到 Hub-and-Spoke，Blackboard 用于高级场景

---

# 协调策略

<div class="columns">
<div>

### 中心化编排
```
Orchestrator
 ├─ 分配任务
 ├─ 控制顺序
 └─ 合成结果
```
✅ 可控、易监控、易审计
❌ 单点复杂性、扩展受限

</div>
<div>

### 去中心化 / 涌现式
```
Agent A ◀──▶ Agent B
  ▲               ▲
  └───▶ Agent C ──┘
```
✅ 高鲁棒性、天然分布式
❌ 设计复杂、调试困难

### Hybrid（推荐）
中心化编排 + 局部自治
平衡可控性与弹性

</div>
</div>

---

---
## Part 3 · Hub-and-Spoke 模式（asyncio 并行）

Orchestrator 用 `asyncio.gather` **并行**调用多个 Agent，再合成结果。

```
用户
 │
 ▼
Orchestrator
 ├─── asyncio.gather ──▶ RetrieverAgent  (0.3s)
 │                  ──▶ MetadataAgent   (0.1s)  ← 同时运行！
 │                  ──▶ SummaryAgent    (0.2s)
 │         ▼ 全部完成后
 └─── GeneratorAgent（拿到所有结果再生成）
```

In [5]:
# ── 异步 Agent 定义 ──────────────────────────────────────────────────────────
async def async_retrieve(query: str) -> list[dict]:
    await asyncio.sleep(0.3)   # 模拟 I/O 延迟
    return fake_retrieve(query, top_k=5, latency=0)

async def async_metadata(query: str) -> dict:
    """模拟获取文档元数据"""
    await asyncio.sleep(0.1)
    return {"doc_count": random.randint(100, 500), "last_updated": "2025-01"}

async def async_summary_hint(query: str) -> str:
    """模拟摘要提示词生成"""
    await asyncio.sleep(0.2)
    return f"请用简洁语言回答：{query}"

async def orchestrator(query: str) -> dict:
    t0 = time.perf_counter()
    print(f"  [Orchestrator] 并行调用 3 个 Agent...")

    # ── 并行阶段：同时获取检索结果、元数据、提示词 ──────────────────────────
    hits, meta, hint = await asyncio.gather(
        async_retrieve(query),
        async_metadata(query),
        async_summary_hint(query),
    )

    print(f"  [Orchestrator] 并行完成，开始生成...")

    # ── 串行阶段：依赖上面结果的生成步骤 ────────────────────────────────────
    await asyncio.sleep(0.5)   # 模拟生成
    answer = fake_generate(hint, hits, latency=0)

    elapsed = round(time.perf_counter() - t0, 3)
    return {
        "answer": answer,
        "meta": meta,
        "latency_s": elapsed,
        "parallel_savings_s": round((0.3 + 0.1 + 0.2) - max(0.3, 0.1, 0.2), 3),
    }

# ── 运行 ─────────────────────────────────────────────────────────────────────
result = asyncio.run(orchestrator("什么是向量数据库？"))
print(f"\n总延迟：{result['latency_s']}s")
print(f"并行节省：{result['parallel_savings_s']}s（相比串行）")
print(f"文档数：{result['meta']['doc_count']}")

  [Orchestrator] 并行调用 3 个 Agent...
  [Orchestrator] 并行完成，开始生成...

总延迟：0.81s
并行节省：0.3s（相比串行）
文档数：107


> **关键观察**：
> - 3 个 Agent 并行，总时间 ≈ `max(0.3, 0.1, 0.2) + 0.5 = 0.8s`
> - 串行则需 `0.3 + 0.1 + 0.2 + 0.5 = 1.1s`
> - `asyncio.gather` 是实现并行的关键

## 📖 讲义

# Part 3
## 通信机制与状态管理

---

# 四种通信机制

| 机制 | 代表技术 | 延迟 | 耦合度 | 适用场景 |
|------|----------|------|--------|----------|
| **Sync RPC** | HTTP / gRPC | 低 | 高 | 实时决策、简单编排 |
| **Async Queue** | Redis / Kafka / SQS | 中 | 低 | 耗时操作、高并发 |
| **Pub-Sub** | SNS / Kafka Topic | 中 | 很低 | 广播事件、多订阅方 |
| **Shared Storage** | S3 + DB + 信号 | 高 | 很低 | 大文件、artifacts 共享 |

---

# 选择通信机制的决策树

```
需要立即拿到结果？
 │
 ├─ YES → 延迟是否 < 100ms？
 │          ├─ YES → gRPC / HTTP
 │          └─ NO  → HTTP + polling / WebSocket
 │
 └─ NO → 任务是否耗时 > 1s？
           ├─ YES → Message Queue（Redis / Kafka）
           └─ NO  → 可以用 async HTTP + callback
```

**黄金法则**
- 用户直接等待 → 同步 RPC
- 后台批处理 → 异步队列
- 广播通知 → Pub-Sub
- 大文件传递 → Shared Storage

---

# 状态、记忆与幂等

<div class="columns">
<div>

### Agent State 分类

**短期状态（任务内）**
- 随消息一起传递
- 无需持久化
- 示例：当前检索结果

**长期状态（跨会话）**
- 存 DB / Redis
- 需要 TTL 和生命周期管理
- 示例：用户偏好、对话历史

</div>
<div>

### 幂等设计（关键！）

```python
# 每条消息带唯一 ID
job = {
    "request_id": "uuid-xxx",
    "query": "...",
}

# 处理前检查是否已处理
if redis.get(f"done:{job['request_id']}"):
    return  # 跳过，防止重复

process(job)
redis.setex(f"done:{job['request_id']}", 3600, 1)
```

**为什么必须幂等？**
网络重试 + 消费者崩溃恢复都会重复投递消息

</div>
</div>

---

# Backpressure 与流量控制

```
生产者速率 >> 消费者速率 → 队列积压 → OOM / 延迟飙升
```

**应对策略**

| 策略 | 实现 | 效果 |
|------|------|------|
| 队列长度限制 | `maxmemory-policy allkeys-lru` | 防止内存耗尽 |
| 消费者并发控制 | Worker 数量 × 每个 Worker 并发数 | 精细控制吞吐 |
| 速率限制 | Token bucket / Redis rate limiter | 保护下游 |
| 背压信号 | 队列长度 > 阈值时拒绝新请求 | 快速失败优于慢挂 |
| Autoscaling | HPA 基于队列长度扩缩 Worker | 弹性应对流量波动 |

---

---
## Part 4 · Blackboard 模式（共享状态 + 事件）

多个 Agent 通过**共享黑板**（字典 + `threading.Event`）协作，无需直接互调。

```
                  写入         订阅事件
IngestAgent   ──▶ bb["chunks"] → event: chunks_ready
RetrieverAgent ──▶ bb["hits"]  → event: hits_ready
RerankerAgent  ──▶ bb["topk"]  → event: topk_ready
GeneratorAgent ──▶ bb["answer"]→ event: done
```

In [6]:
# ── Blackboard 实现 ──────────────────────────────────────────────────────────
class Blackboard:
    def __init__(self):
        self._data: dict[str, Any] = {}
        self._events: dict[str, threading.Event] = defaultdict(threading.Event)
        self._lock = threading.Lock()

    def write(self, key: str, value: Any):
        with self._lock:
            self._data[key] = value
        print(f"  📋 Blackboard.write({key!r})")
        self._events[key].set()          # 通知订阅者

    def read(self, key: str) -> Any:
        with self._lock:
            return self._data.get(key)

    def wait_for(self, key: str, timeout: float = 5.0) -> bool:
        """阻塞直到 key 被写入"""
        return self._events[key].wait(timeout=timeout)


# ── 各 Agent 定义（每个都是独立线程）────────────────────────────────────────
def ingest_agent(bb: Blackboard, query: str):
    print("  [Ingest] 开始分片...")
    time.sleep(0.1)
    bb.write("query", query)
    bb.write("chunks", [{"text": t} for t in CORPUS[:4]])

def retriever_agent(bb: Blackboard):
    bb.wait_for("chunks")
    query  = bb.read("query")
    chunks = bb.read("chunks")
    print("  [Retriever] 检索中...")
    time.sleep(0.3)
    hits = [
        {"text": c["text"], "score": round(random.uniform(0.5, 1.0), 3)}
        for c in sorted(chunks, key=lambda _: random.random())[:3]
    ]
    bb.write("hits", hits)

def reranker_agent(bb: Blackboard):
    bb.wait_for("hits")
    query = bb.read("query")
    hits  = bb.read("hits")
    print("  [Reranker] 重排中...")
    time.sleep(0.2)
    topk = fake_rerank(query, hits, top_k=2, latency=0)
    bb.write("topk", topk)

def generator_agent(bb: Blackboard):
    ok = bb.wait_for("topk", timeout=5.0)
    query = bb.read("query")
    topk  = bb.read("topk") or []
    print("  [Generator] 生成中...")
    answer = fake_generate(query, topk, latency=0.5)
    bb.write("answer", answer)


# ── 运行：启动所有 Agent，让它们通过黑板协作 ─────────────────────────────────
bb = Blackboard()
t0 = time.perf_counter()

threads = [
    threading.Thread(target=ingest_agent,    args=(bb, "什么是 Blackboard 模式？")),
    threading.Thread(target=retriever_agent, args=(bb,)),
    threading.Thread(target=reranker_agent,  args=(bb,)),
    threading.Thread(target=generator_agent, args=(bb,)),
]
for t in threads:
    t.start()
for t in threads:
    t.join()

print(f"\n⏱ 总耗时：{round(time.perf_counter()-t0, 3)}s")
print(f"📝 最终答案：{bb.read('answer')[:80]}...")

  [Ingest] 开始分片...
  📋 Blackboard.write('query')
  📋 Blackboard.write('chunks')
  [Retriever] 检索中...
  📋 Blackboard.write('hits')
  [Reranker] 重排中...
  📋 Blackboard.write('topk')
  [Generator] 生成中...
  📋 Blackboard.write('answer')

⏱ 总耗时：1.112s
📝 最终答案：[答案] 关于「什么是 Blackboard 模式？」：幂等设计让系统在重试时不产生副作用。 向量数据库支持语义相似度搜索。......


> **关键观察**：
> - 各 Agent **不互相调用**，只通过黑板读写
> - `wait_for` 实现事件驱动，无需轮询
> - 新增一个 Agent 只需监听相应字段，**不修改任何已有代码**

## 📖 讲义

# Part 4
## 三个动手示例

---

# 示例概览

| 示例 | 模式 | 技术栈 | 难度 |
|------|------|--------|------|
| **示例 1** | Pipeline | Redis + Python Workers | ⭐ 入门 |
| **示例 2** | Hub-and-Spoke | FastAPI + HTTP | ⭐⭐ 进阶 |
| **示例 3** | Blackboard | DB + Pub-Sub | ⭐⭐⭐ 高级 |

> 每个示例均可在本地独立运行，课后建议按顺序动手实现

---

# 示例 1：Redis Pipeline（架构）

```
用户请求
   │
   ▼
Orchestrator ──push──▶ jobs:retriever
                              │
                        RetrieverWorker
                              │ push
                              ▼
                        jobs:reranker
                              │
                        RerankerWorker
                              │ push
                              ▼
                        jobs:generator
                              │
                        GeneratorWorker ──write──▶ results:{job_id}
                                                        │
                                                        ▼
                                                  API 轮询取结果
```

---

# 示例 1：核心代码

**Orchestrator — 提交任务**
```python
# orchestrator.py
import redis, json, uuid

r = redis.Redis()

def submit_query(query: str) -> str:
    job_id = str(uuid.uuid4())
    job = {"request_id": job_id, "query": query, "stage": "retriever"}
    r.lpush("jobs:retriever", json.dumps(job))
    return job_id
```

**Retriever Worker — 消费 & 转发**
```python
# retriever_worker.py
while True:
    _, data = r.brpop("jobs:retriever")   # blocking pop
    job = json.loads(data)

    # 幂等检查
    if r.get(f"done:retriever:{job['request_id']}"):
        continue

    job["hits"] = do_retrieval(job["query"], top_k=20)
    job["stage"] = "reranker"
    r.lpush("jobs:reranker", json.dumps(job))
    r.setex(f"done:retriever:{job['request_id']}", 3600, 1)
```

---

# 示例 1：Reranker & Generator

**Reranker Worker**
```python
# reranker_worker.py
while True:
    _, data = r.brpop("jobs:reranker")
    job = json.loads(data)
    job["topk"] = cross_encoder_rerank(job["query"], job["hits"], top_k=5)
    job["stage"] = "generator"
    r.lpush("jobs:generator", json.dumps(job))
```

**Generator Worker**
```python
# generator_worker.py
while True:
    _, data = r.brpop("jobs:generator")
    job = json.loads(data)
    context = "\n".join([h["text"] for h in job["topk"]])
    answer = llm_generate(question=job["query"], context=context)
    r.setex(f"result:{job['request_id']}", 3600, json.dumps({"answer": answer}))
```

> **运行方式**：分别在 3 个终端启动三个 Worker，再调用 `submit_query()`

---

# 示例 2：HTTP Orchestrator（架构）

```
用户
 │ POST /query
 ▼
┌─────────────────────────────────┐
│         Orchestrator            │
│  1. POST /retrieve → hits       │
│  2. POST /rerank   → topk       │
│  3. POST /generate → answer     │
│  4. POST /verify   → verdict    │
└─────────────────────────────────┘
     │         │         │         │
 :8001      :8002     :8003     :8004
Retriever  Reranker Generator Verifier
```

---

# 示例 2：Orchestrator 代码

```python
# orchestrator_http.py
import httpx
from fastapi import FastAPI

app = FastAPI()
SERVICES = {
    "retriever": "http://localhost:8001",
    "reranker":  "http://localhost:8002",
    "generator": "http://localhost:8003",
    "verifier":  "http://localhost:8004",
}

@app.post("/query")
async def handle_query(payload: dict):
    q = payload["query"]
    async with httpx.AsyncClient(timeout=30) as client:
        hits    = (await client.post(f"{SERVICES['retriever']}/retrieve",
                                      json={"query": q})).json()["hits"]
        topk    = (await client.post(f"{SERVICES['reranker']}/rerank",
                                      json={"query": q, "candidates": hits})).json()["topk"]
        answer  = (await client.post(f"{SERVICES['generator']}/generate",
                                      json={"question": q, "context": topk})).json()["answer"]
        verdict = (await client.post(f"{SERVICES['verifier']}/check",
                                      json={"answer": answer, "context": topk})).json()
    return {"answer": answer, "verdict": verdict}
```

> 每个微服务独立用 FastAPI 实现，用 Docker Compose 一键启动

---

# 示例 3：Blackboard 模式

**核心思路**：共享数据库作为"黑板"，Agent 通过事件感知变化

```python
# blackboard.py — 简化示意
import redis

r = redis.Redis()

class Blackboard:
    def write(self, task_id: str, field: str, value):
        """写入字段并发布变更事件"""
        r.hset(f"task:{task_id}", field, json.dumps(value))
        r.publish(f"task:{task_id}:events", field)

    def read(self, task_id: str, field: str):
        raw = r.hget(f"task:{task_id}", field)
        return json.loads(raw) if raw else None

    def subscribe(self, task_id: str):
        """订阅该任务的所有变更"""
        pubsub = r.pubsub()
        pubsub.subscribe(f"task:{task_id}:events")
        return pubsub
```

**流程**：Ingest Agent 写 `chunks` → Retriever 订阅 `new_query` 事件并写 `hits` → Reranker 订阅 `hits_ready` 并写 `topk` → Generator 读取 `topk` 生成并写 `answer`

---

---
## Part 5 · 幂等设计（Idempotency）

**问题**：消息队列在网络故障或消费者崩溃后会**重复投递**，如果处理不是幂等的，就会产生重复写入或重复 API 调用。

**解决方案**：每条消息带 `request_id`，处理前检查是否已处理过。

In [7]:
# ── 模拟幂等处理器 ───────────────────────────────────────────────────────────
class IdempotentProcessor:
    def __init__(self):
        self._processed: set[str] = set()    # 生产中用 Redis SET + TTL
        self._call_count = 0

    def process(self, job: dict) -> dict | None:
        rid = job["request_id"]

        # 幂等检查
        if rid in self._processed:
            print(f"  ⚠️  [{rid[:8]}] 重复消息，跳过（已处理过）")
            return None

        self._call_count += 1
        self._processed.add(rid)
        result = fake_generate(job["query"], fake_retrieve(job["query"], latency=0), latency=0)
        print(f"  ✅ [{rid[:8]}] 处理完成（第 {self._call_count} 次实际处理）")
        return {"result": result}


processor = IdempotentProcessor()

# ── 模拟场景：同一条消息被投递 3 次（网络重试）───────────────────────────────
job = {"request_id": str(uuid.uuid4()), "query": "幂等性是什么？"}

print("=== 模拟消息重复投递 ===")
for i in range(3):
    print(f"\n投递第 {i+1} 次：")
    processor.process(job)

print(f"\n实际处理次数：{processor._call_count}（应为 1，不是 3）")

# ── 不同 request_id 的新消息正常处理 ─────────────────────────────────────────
print("\n=== 新消息（不同 request_id）===")
for q in ["什么是 RAG？", "什么是向量数据库？"]:
    new_job = {"request_id": str(uuid.uuid4()), "query": q}
    processor.process(new_job)

print(f"\n总实际处理次数：{processor._call_count}（3 条唯一消息）")

=== 模拟消息重复投递 ===

投递第 1 次：
  ✅ [7b557029] 处理完成（第 1 次实际处理）

投递第 2 次：
  ⚠️  [7b557029] 重复消息，跳过（已处理过）

投递第 3 次：
  ⚠️  [7b557029] 重复消息，跳过（已处理过）

实际处理次数：1（应为 1，不是 3）

=== 新消息（不同 request_id）===
  ✅ [8d9d4594] 处理完成（第 2 次实际处理）
  ✅ [67954af5] 处理完成（第 3 次实际处理）

总实际处理次数：3（3 条唯一消息）


## 📖 讲义

# Part 5
## 生产级要点

---

# 可观测性：三大支柱

<div class="columns">
<div>

### Metrics（指标）
```python
from prometheus_client import Counter, Histogram

requests_total = Counter(
    "agent_requests_total",
    "Total requests",
    ["agent_name", "status"]
)
latency = Histogram(
    "agent_latency_seconds",
    "Request latency",
    ["agent_name"]
)

# 使用
with latency.labels("retriever").time():
    hits = do_retrieval(query)
requests_total.labels("retriever", "success").inc()
```

</div>
<div>

### Traces（链路追踪）
```python
from opentelemetry import trace

tracer = trace.get_tracer("retriever")

with tracer.start_as_current_span("retrieve") as span:
    span.set_attribute("query", query)
    span.set_attribute("top_k", 20)
    hits = do_retrieval(query)
    span.set_attribute("hits_count", len(hits))
```

### Logs（结构化日志）
```python
import structlog
log = structlog.get_logger()

log.info("retrieval_done",
    request_id=job["request_id"],
    query=query,
    hits=len(hits),
    latency_ms=elapsed)
```

</div>
</div>

---

# 安全隔离与最小权限

**Agent 安全清单**

| 层次 | 措施 |
|------|------|
| **网络** | 各 Agent 只开放必要端口，使用 mTLS |
| **IAM** | 每个 Agent 独立 Service Account，最小权限 |
| **输入校验** | 所有入参用 Pydantic / jsonschema 校验 |
| **输出过滤** | LLM 输出过滤 Prompt Injection / PII |
| **速率限制** | 每个 Agent 入口限流（令牌桶） |
| **审计日志** | 记录所有 tool call 与外部 API 调用 |

> **关键原则**：绝不直接执行 LLM 生成的命令，必须经过参数白名单校验

---

# 成本优化策略

<div class="columns">
<div>

### 分层模型路由
```python
def select_model(task_type: str) -> str:
    routing = {
        "retrieval_score":  "haiku-4-5",   # 便宜
        "rerank":           "haiku-4-5",
        "generation_simple":"sonnet-4-6",  # 主力
        "generation_complex":"opus-4-6",   # 贵，按需
        "verification":     "haiku-4-5",
    }
    return routing.get(task_type, "sonnet-4-6")
```

</div>
<div>

### 缓存策略
```python
import hashlib

def cached_retrieve(query: str):
    key = f"cache:retrieve:{hashlib.md5(query.encode()).hexdigest()}"
    cached = redis.get(key)
    if cached:
        return json.loads(cached)
    result = do_retrieval(query)
    redis.setex(key, 3600, json.dumps(result))
    return result
```

### 批处理
- 聚合多个请求一次调用 LLM
- 使用 Embedding batch API

</div>
</div>

---

# Graceful Degradation（优雅降级）

```python
# orchestrator_with_fallback.py
async def handle_query_with_fallback(query: str):
    try:
        # 正常路径：完整 Pipeline
        hits  = await retriever.retrieve(query)
        topk  = await reranker.rerank(query, hits)   # 可能失败
        answer = await generator.generate(query, topk)
    except RerankerUnavailable:
        # 降级：跳过 Reranker，直接用粗检结果
        logger.warning("reranker_down_fallback", query=query)
        answer = await generator.generate(query, hits[:5])
    except GeneratorTimeout:
        # 再降级：直接返回检索片段
        return {"answer": None, "snippets": hits[:3], "degraded": True}

    return {"answer": answer, "degraded": False}
```

**降级层次**：完整 Pipeline → 跳过 Reranker → 仅返回检索片段 → 缓存结果 → 静态兜底

---

---
## Part 6 · 可观测性（Metrics + Trace）

生产中用 Prometheus + OpenTelemetry，这里用纯 Python 模拟相同概念：
- **Metrics**：计数器、直方图（延迟分布）
- **Trace**：全链路 request_id，记录每个 Agent 的耗时与结果

In [8]:
import statistics

# ── 简易 Metrics 收集器 ────────────────────────────────────────────────────
class Metrics:
    def __init__(self):
        self.counters: dict[str, int]       = defaultdict(int)
        self.histograms: dict[str, list]    = defaultdict(list)

    def inc(self, name: str, labels: dict = {}):
        key = name + str(sorted(labels.items()))
        self.counters[key] += 1

    def observe(self, name: str, value: float, labels: dict = {}):
        key = name + str(sorted(labels.items()))
        self.histograms[key].append(value)

    def summary(self):
        print("\n📊 Metrics Summary")
        print("─" * 50)
        for k, v in self.counters.items():
            print(f"  counter  {k} = {v}")
        for k, vals in self.histograms.items():
            p50 = statistics.median(vals)
            p95 = sorted(vals)[int(len(vals)*0.95)] if len(vals) > 1 else vals[0]
            print(f"  hist     {k}  p50={p50:.3f}s  p95={p95:.3f}s  n={len(vals)}")

metrics = Metrics()

# ── 带 Trace 和 Metrics 的 Pipeline ─────────────────────────────────────────
def traced_pipeline(query: str) -> dict:
    trace_id = str(uuid.uuid4())[:8]
    trace_log = []

    def span(name: str, fn, *args, **kwargs):
        t = time.perf_counter()
        result = fn(*args, **kwargs)
        elapsed = round(time.perf_counter() - t, 3)
        trace_log.append({"span": name, "latency_s": elapsed})
        metrics.observe("agent_latency_seconds", elapsed, {"agent": name})
        metrics.inc("agent_requests_total", {"agent": name, "status": "ok"})
        return result

    hits    = span("retriever", fake_retrieve, query, latency=0.3)
    topk    = span("reranker",  fake_rerank,   query, hits, latency=0.2)
    answer  = span("generator", fake_generate, query, topk, latency=0.5)
    verdict = span("verifier",  fake_verify,   answer, topk, latency=0.15)

    print(f"\n🔍 Trace [{trace_id}] for: {query[:30]}")
    for s in trace_log:
        bar = "█" * int(s["latency_s"] * 20)
        print(f"  {s['span']:<12} {bar:<20} {s['latency_s']}s")

    return {"answer": answer, "verdict": verdict, "trace_id": trace_id}

# ── 跑 5 次，生成 metrics ────────────────────────────────────────────────────
for q in ["RAG 是什么？", "幂等设计？", "可观测性？", "Blackboard？", "Pipeline？"]:
    traced_pipeline(q)

metrics.summary()


🔍 Trace [32dd9101] for: RAG 是什么？
  retriever    ██████               0.3s
  reranker     ████                 0.2s
  generator    ██████████           0.5s
  verifier     ███                  0.15s

🔍 Trace [bc402c63] for: 幂等设计？
  retriever    ██████               0.301s
  reranker     ████                 0.2s
  generator    ██████████           0.501s
  verifier     ███                  0.15s

🔍 Trace [88428d41] for: 可观测性？
  retriever    ██████               0.301s
  reranker     ████                 0.202s
  generator    ██████████           0.501s
  verifier     ███                  0.15s

🔍 Trace [ba54c293] for: Blackboard？
  retriever    ██████               0.3s
  reranker     ████                 0.2s
  generator    ██████████           0.5s
  verifier     ███                  0.15s

🔍 Trace [d877d384] for: Pipeline？
  retriever    ██████               0.301s
  reranker     ████                 0.201s
  generator    ██████████           0.5s
  verifier     ███                 

## 📖 讲义

# Part 6
## 课堂练习 & 总结

---

# 课堂练习（五个阶段）

| 阶段 | 任务 | 目标 |
|------|------|------|
| **1. 入门** | 实现 Redis Pipeline 三 Agent（Retriever → Generator → Verifier），跑通端到端并打印 trace | 理解消息传递基础 |
| **2. 进阶** | 把 Reranker 抽成独立 HTTP 服务，用 Docker Compose 编排，压测并观察延迟 | 掌握微服务拆分 |
| **3. 监控** | 为每个 Agent 加 Prometheus 指标 + p95 告警，演练 Reranker 下线降级 | 生产可观测性 |
| **4. 安全** | 实现 Pydantic 入参校验 + 参数白名单，模拟恶意参数并记录拒绝事件 | 安全意识 |
| **5. 评估** | 接入 RAGAS 运行 `rag_e2e_eval.py`，保存多次版本的评估结果对比 | 质量迭代意识 |

---

# 常见陷阱速查

| 陷阱 | 典型症状 | 对策 |
|------|----------|------|
| 单点瓶颈 | Orchestrator CPU 100%，其他 Agent 空闲 | 拆分职责，引入异步队列 |
| 共享可变状态 | 随机数据错乱，难以复现 | 乐观锁 / 事件溯源 |
| 缺少幂等 | 重试导致重复写入或重复 API 调用 | request-id + 处理记录 |
| 无监控 | 出问题后无法定位根因 | Metrics + Traces + 结构化日志 |
| 无限循环 | 资源耗尽，成本暴增 | 最大调用深度 + 成本预算 |
| 过度信任 LLM | 执行了恶意 tool call | 参数白名单 + 人工审批 |
| 忽视隐私 | 敏感数据在多 Agent 间明文传播 | PII 脱敏 + 访问控制 |

---

# 课程小结

**Multi-Agent Architecture 的核心价值**

```
复杂任务 ──拆分──▶ 多个职责明确的 Agent ──协作──▶ 可扩展的生产系统
```

**三个必记原则**
1. **职责分离 + 松耦合** — Agent 之间只通过接口（消息/API）交互
2. **幂等 + 可观测** — 每条消息有 ID，每个 Agent 暴露 metrics/traces
3. **降级 + 安全** — 提前设计失败路径，永不直接执行 LLM 输出

**技术演进路径**
```
单体 Agent
  → Pipeline（Redis Queue）
    → Hub-and-Spoke（HTTP 微服务）
      → Blackboard（事件驱动）
        → 混合架构（按需组合）
```

---

# 推荐资源

<div class="columns">
<div>

### 框架 & 工具
- **LangChain / LangGraph** — Multi-Agent patterns
- **CrewAI** — Role-based agents
- **AutoGen** — Microsoft multi-agent framework
- **Haystack** — Production RAG pipelines

### 消息队列
- **Redis Streams** — 入门首选
- **RabbitMQ** — 经典 AMQP
- **Apache Kafka** — 高吞吐场景

</div>
<div>

### 可观测性
- **LangSmith / Langfuse** — LLM tracing
- **OpenTelemetry** — 标准 traces
- **Prometheus + Grafana** — Metrics

### 部署
- **Docker Compose** — 本地多服务
- **Kubernetes + HPA** — 生产弹性扩缩
- **Pinecone / Qdrant / OpenSearch** — 向量数据库

### 评估
- **RAGAS** — RAG 质量评估框架

</div>
</div>

---

---
## Part 7 · 优雅降级（Graceful Degradation）

**场景**：Reranker 服务不可用时，系统不应整体崩溃，而应**降级**继续提供服务。

```
正常路径：  retrieve → rerank → generate  ✅
降级路径1： retrieve → (skip rerank) → generate  ⚠️ 质量略降
降级路径2： (retrieve timeout) → return cached snippets  ⚠️ 更大降级
兜底路径：  return static fallback message  🆘
```

In [9]:
class RerankerUnavailable(Exception): pass
class RetrieverTimeout(Exception): pass

class ResilientOrchestrator:
    def __init__(self, reranker_up: bool = True, retriever_up: bool = True):
        self.reranker_up  = reranker_up
        self.retriever_up = retriever_up
        self._cache: dict[str, list] = {}

    def handle(self, query: str) -> dict:
        degraded_reason = None

        # ── 检索（有缓存则用缓存）────────────────────────────────────────────
        try:
            if not self.retriever_up:
                raise RetrieverTimeout("retriever unavailable")
            hits = fake_retrieve(query, latency=0.1)
            self._cache[query] = hits          # 写缓存
        except RetrieverTimeout:
            hits = self._cache.get(query, [])
            if hits:
                degraded_reason = "retriever_timeout_used_cache"
            else:
                return {
                    "answer": "暂时无法检索，请稍后重试。",
                    "degraded": True,
                    "reason": "retriever_down_no_cache",
                }

        # ── 重排（降级：跳过，直接用粗检结果）───────────────────────────────
        try:
            if not self.reranker_up:
                raise RerankerUnavailable("reranker is down")
            topk = fake_rerank(query, hits, top_k=3, latency=0.1)
        except RerankerUnavailable:
            topk = hits[:3]                    # 降级：直接用 top-3 粗检结果
            degraded_reason = degraded_reason or "reranker_down_using_raw_hits"

        # ── 生成 ─────────────────────────────────────────────────────────────
        answer = fake_generate(query, topk, latency=0.3)
        return {
            "answer": answer[:80],
            "degraded": degraded_reason is not None,
            "reason": degraded_reason,
        }


# ── 测试三种场景 ──────────────────────────────────────────────────────────────
scenarios = [
    ("✅ 正常路径",          dict(reranker_up=True,  retriever_up=True)),
    ("⚠️  Reranker 故障",   dict(reranker_up=False, retriever_up=True)),
    ("🆘 Retriever 故障",   dict(reranker_up=True,  retriever_up=False)),
]

query = "什么是优雅降级？"
for label, flags in scenarios:
    orc = ResilientOrchestrator(**flags)
    r   = orc.handle(query)
    status = "降级" if r["degraded"] else "正常"
    print(f"{label}")
    print(f"  状态：{status}  原因：{r.get('reason', 'none')}")
    print(f"  答案：{r['answer'][:60]}...")
    print()

✅ 正常路径
  状态：正常  原因：None
  答案：[答案] 关于「什么是优雅降级？」：消息队列解耦生产者和消费者，支持异步处理。 Blackboard 模式通过共享工作区...

⚠️  Reranker 故障
  状态：降级  原因：reranker_down_using_raw_hits
  答案：[答案] 关于「什么是优雅降级？」：消息队列解耦生产者和消费者，支持异步处理。 向量数据库支持语义相似度搜索。......

🆘 Retriever 故障
  状态：降级  原因：retriever_down_no_cache
  答案：暂时无法检索，请稍后重试。...



> **关键观察**：
> - Reranker 故障时，系统仍能返回答案（质量略降）
> - Retriever 故障时，优先用缓存；无缓存才返回兜底消息
> - 每个降级路径都有明确的 `reason`，便于监控告警

## 📖 讲义

# Q & A

## 有什么问题？

<br>

**课后作业**：完成练习阶段 1（Redis Pipeline 三 Agent）
提交：代码仓库链接 + 一张端到端 trace 截图

<br>

*下一课：Multi-Agent 评估与持续优化*

---
## 课程小结

| 模式 | 核心机制 | 适用场景 | Demo 实现 |
|------|----------|----------|-----------|
| **Monolithic** | 串行函数调用 | 简单原型 | `monolithic_agent()` |
| **Pipeline** | `queue.Queue` + 线程 Worker | 线性流水、高并发 | `q_retrieve / q_rerank / q_generate` |
| **Hub-and-Spoke** | `asyncio.gather` 并行 | 多角色协作、并行加速 | `orchestrator()` |
| **Blackboard** | 共享字典 + `threading.Event` | 迭代协作、松耦合 | `Blackboard` 类 |
| **幂等** | `request_id` + 已处理集合 | 所有消息驱动场景 | `IdempotentProcessor` |
| **可观测性** | `span()` + `Metrics` | 生产监控、性能诊断 | `traced_pipeline()` |
| **优雅降级** | `try/except` + 降级路径 | 高可用系统 | `ResilientOrchestrator` |

### 三条铁律

1. **每条消息带 `request_id`**，处理前先做幂等检查
2. **每个 Agent 暴露 latency + status**，不监控等于盲飞
3. **提前设计失败路径**，不要等故障发生后再想降级逻辑

---
*下一步：把这些 Agent 换成真实的 LLM API 调用，并接入 Redis / Kafka 消息队列*